# Dennemeyer HTML Tools — Builder

Regenerates the IPscore and NPV Target Planner HTML tools in this folder from
structured data + shared Jinja2 templates, instead of hand-editing the HTML directly.

**Why this exists**: `BUILD_LOG.md` documents how these tools were originally built, but
every fix since has been a direct hand-edit of the HTML — which is exactly how the
apostrophe-`SyntaxError` bug in `IPscore_IT.html` happened (an Italian apostrophe inside
a single-quoted JS string silently killed the whole `<script>` block). This notebook makes
the build reproducible and adds an automated check for that exact class of bug.

**What this is *not***: an Excel → HTML generator. `IPscore_3.01.xlsx` only contains the
deterministic skeleton (question IDs, plain EPO English text, risk/opportunity flags, OEK
values). The Italian translations, the rich € benchmark help text, all three demo
narratives (MedTech Italia / BioSense Technologies / NovaMed Diagnostics), and the NPV
Planner's plain-language insights were authored directly in the HTML in past sessions —
none of that exists in the Excel. So this pipeline is: **extract already-authored content
out of the current HTML → structured JSON → re-assemble through a template.** The current
HTML files are the source of truth for *content*; this notebook is the source of truth for
*assembly*.

**Scope**: IPscore family (`IPscore_IT.html`, `IPscore_IT_Demo.html`, `IPscore_EN_Demo.html`)
+ NPV Target Planner family (`NPV_Target_Planner_EN.html`, `NPV_Target_Planner_IT.html`).
The ASP Invention Assessment tools are a separate family (different source Excel) and are
out of scope here.

## Pipeline

```
Dennemeyer/*.html (current, hand-authored)
        │  extract_vars.js / extract_qualcards.js / extract_oekcards.js / extract_filldemo.js
        │  (Node VM sandbox: evaluates the real <script> so data comes out faithfully,
        │   not retyped by hand — 40 questions × 2 languages, 8 OEK cards, 6 qual cards, etc.)
        ▼
build/data/*.json (per-language content packs: questions, UI strings, demo scenarios)
        │  render.py (Jinja2)
        ▼
build/templates/*.html.j2  +  build/data/*.json
        │
        ▼
build/dist/*.html  (rendered output — reviewed here before touching the live files)
```


In [1]:
import sys, json, subprocess
from pathlib import Path

BUILD = Path.cwd() / "build"
assert BUILD.is_dir(), "Run this notebook from /home/jovyan/Dennemeyer"
sys.path.insert(0, str(BUILD))

import importlib
import render as render_mod
importlib.reload(render_mod)

print("jinja2 templates:", [p.name for p in (BUILD / "templates").glob("*.j2")])
print("data packs:", [p.name for p in (BUILD / "data").glob("*.json")])


jinja2 templates: ['ipscore_template.html.j2', 'npv_planner_template.html.j2']
data packs: ['ipscore_demo_en.json', 'npv_demo_it.json', 'npv_ui_strings_it.json', 'ipscore_questions_en.json', 'ipscore_demo_it.json', 'npv_insights_it.json', 'npv_ui_strings_en.json', 'ipscore_ui_strings_it.json', 'npv_demo_en.json', 'npv_insights_en.json', 'ipscore_ui_strings_en.json', 'ipscore_questions_it.json']


## 1. Extraction (already run once — re-run only if the *live* HTML files change)

The four `extract_*.js` scripts in `build/` pull structured data out of the current
`Dennemeyer/*.html` files by running their real `<script>` block in a sandboxed Node VM
(DOM calls stubbed to no-ops) — this is faithful extraction, not manual retyping:

- `extract_vars.js` — top-level JS object/array literals (`QS`, `INSIGHTS`, `PARAM_META`, `PARAM_TIPS`, `OEK_VALUES`, `COLORS`, `STD5`, `SEC_NAMES`, ...)
- `extract_qualcards.js` — intercepts the 6 `qualCard(id, text, opts)` calls inside `buildPages()`
- `extract_oekcards.js` — intercepts the 8 `oekCard(id, icon, title, question, context, opts)` calls
- `extract_filldemo.js` — runs `fillDemo()` and records every `getElementById(id).value = ...` and `onRadio`/`selectOEK`/`selectQual` call

Re-run this cell only if you hand-edit one of the *live* `IPscore_*.html` /
`NPV_Target_Planner_*.html` files and want to re-import that change into `build/data/`
before switching to editing the JSON going forward.


In [2]:
# Re-extraction is opt-in — flip to True only if you've hand-edited a live HTML file
# and want to re-import its content into build/data/ before switching to JSON-based edits.
RUN_EXTRACTION = False

if RUN_EXTRACTION:
    import subprocess
    def run(cmd):
        r = subprocess.run(cmd, cwd=BUILD, capture_output=True, text=True)
        print(' '.join(cmd), '->', 'OK' if r.returncode == 0 else 'FAILED')
        if r.returncode != 0:
            print(r.stderr)
        return r.stdout

    # Example: re-extract IPscore IT questions after a hand-edit
    # out = run(["node", "extract_vars.js", "../IPscore_IT.html", "QS"])
    # json.dump(json.loads(out)["QS"], open(BUILD/"data"/"ipscore_questions_it.json","w"),
    #           ensure_ascii=False, indent=2)
    print("Set RUN_EXTRACTION=True and uncomment the specific extraction you need.")
else:
    print("Skipped — build/data/*.json already holds the current extracted content.")


Skipped — build/data/*.json already holds the current extracted content.


## 2. Templates

`build/templates/ipscore_template.html.j2` and `build/templates/npv_planner_template.html.j2`
were built once (`build/make_ipscore_template.py`, `build/make_npv_template.py`) by taking a
real working HTML file and applying **uniqueness-checked literal substitutions**: each
hardcoded string is asserted to occur exactly once in the source before being replaced by a
Jinja placeholder. That guarantees the template is structurally identical to a file that is
already known to work — only the enumerated language/demo-specific spots become placeholders.
CSS and all engine logic (VAN calculation, radar drawing, navigation, scoring) are untouched
and shared by every rendered variant.

Re-run the two `make_*_template.py` scripts only if you need to change the template
*structure* itself (e.g. adding a new page). For everyday content changes (fixing a
translation, adjusting a benchmark figure, tweaking demo numbers), edit the JSON files in
`build/data/` instead and just re-render below.


In [3]:
# Re-run only if you changed a make_*_template.py generator script itself.
REBUILD_TEMPLATES = False

if REBUILD_TEMPLATES:
    for script in ["make_ipscore_template.py", "make_npv_template.py"]:
        r = subprocess.run([sys.executable, script], cwd=BUILD, capture_output=True, text=True)
        print(script, '->', r.stdout.strip() or r.stderr.strip())
else:
    print("Skipped — build/templates/*.j2 already reflect the current generator scripts.")


Skipped — build/templates/*.j2 already reflect the current generator scripts.


## 3. Render

Loads `build/data/*.json`, renders each of the 5 target files through Jinja2, and writes
them to `build/dist/` — **not** over the live files yet, so you can review first.


In [4]:
dist_files = render_mod.main.__wrapped__ if hasattr(render_mod.main, "__wrapped__") else None

# render.py's main() takes CLI args; call the render_* functions directly here so we can
# inspect each result before writing anything.
outputs = {
    "IPscore_IT.html": render_mod.render_ipscore("it", demo_flag=False),
    "IPscore_IT_Demo.html": render_mod.render_ipscore("it", demo_flag=True),
    "IPscore_EN_Demo.html": render_mod.render_ipscore("en", demo_flag=True),
    "NPV_Target_Planner_EN.html": render_mod.render_npv_planner("en"),
    "NPV_Target_Planner_IT.html": render_mod.render_npv_planner("it"),
}

DIST = BUILD / "dist"
DIST.mkdir(exist_ok=True)
for fname, content in outputs.items():
    (DIST / fname).write_text(content, encoding="utf-8")
    print(f"{fname}: {len(content):>7} bytes -> build/dist/{fname}")


IPscore_IT.html:   71074 bytes -> build/dist/IPscore_IT.html
IPscore_IT_Demo.html:   73716 bytes -> build/dist/IPscore_IT_Demo.html
IPscore_EN_Demo.html:   72038 bytes -> build/dist/IPscore_EN_Demo.html
NPV_Target_Planner_EN.html:   56462 bytes -> build/dist/NPV_Target_Planner_EN.html
NPV_Target_Planner_IT.html:   59495 bytes -> build/dist/NPV_Target_Planner_IT.html


## 4. Verify — Node syntax check

The exact check `BUILD_LOG.md` prescribes after any edit that touches JS string literals:
evaluate the `<script>` block as a `Function` body and confirm it parses. This is what
would have caught the apostrophe bug immediately instead of shipping a frozen page.


In [5]:
def node_syntax_check(path):
    script = subprocess.run(
        ["node", "-e", f'''
const fs=require("fs");
const h=fs.readFileSync("{path}","utf8");
const m=h.match(/<script>([\\s\\S]*?)<\\/script>/);
try {{ new Function(m[1]); console.log("OK"); }}
catch(e) {{ console.log("FAIL: " + e.message); process.exitCode = 1; }}
'''],
        capture_output=True, text=True,
    )
    return script.stdout.strip(), script.returncode

all_ok = True
for fname in outputs:
    out, rc = node_syntax_check(DIST / fname)
    print(f"{fname}: {out}")
    all_ok &= (rc == 0)

assert all_ok, "Fix the syntax error above before proceeding — do not promote a broken file."


IPscore_IT.html: OK


IPscore_IT_Demo.html: OK


IPscore_EN_Demo.html: OK


NPV_Target_Planner_EN.html: OK


NPV_Target_Planner_IT.html: OK


## 5. Verify — diff against the live files

A byte-for-byte diff will **not** be empty and that's expected: the injected data blocks
(`QS`, `OEK_CARDS`, `INSIGHTS`, ...) are serialized as single-line JSON by `json.dumps`,
while the original files hand-formatted the same arrays across multiple lines. This cell
reports the diff size and flags anything that *isn't* just that kind of formatting
difference, which would indicate a real content or logic change worth investigating.


In [6]:
import difflib

# A raw line diff will legitimately show hundreds of changed lines: the injected data
# blocks (QS, OEK_CARDS, INSIGHTS, ...) are serialized as single-line JSON, while the
# original files hand-formatted the same arrays across many lines. A line-level diff
# can't distinguish "reformatted" from "actually different", so instead we extract the
# real data structures back out of both files with the same Node VM tooling used to
# build the data packs, and compare them for exact equality -- that's the check that
# actually matters.
DATA_VARS = {
    "IPscore_IT.html": ["QS", "STD5", "SEC_NAMES"],
    "IPscore_IT_Demo.html": ["QS", "STD5", "SEC_NAMES"],
    "IPscore_EN_Demo.html": ["QS", "STD5", "SEC_NAMES"],
    "NPV_Target_Planner_EN.html": ["INSIGHTS", "PARAM_META", "PARAM_TIPS", "OEK_VALUES"],
    "NPV_Target_Planner_IT.html": ["INSIGHTS", "PARAM_META", "PARAM_TIPS", "OEK_VALUES"],
}

def extract(path, varnames):
    r = subprocess.run(["node", "extract_vars.js", str(path), *varnames], cwd=BUILD,
                        capture_output=True, text=True)
    return json.loads(r.stdout)

for fname, varnames in DATA_VARS.items():
    live = Path.cwd() / fname
    if not live.exists():
        print(f"{fname}: no live file yet (new output)")
        continue
    live_data = extract(live, varnames)
    gen_data = extract(DIST / fname, varnames)
    same = live_data == gen_data
    live_lines = live.read_text(encoding="utf-8").splitlines()
    gen_lines = (DIST / fname).read_text(encoding="utf-8").splitlines()
    n_changed = sum(1 for l in difflib.unified_diff(live_lines, gen_lines, lineterm="")
                     if l.startswith(("+", "-")) and not l.startswith(("+++", "---")))
    print(f"{fname}: data identical={same}  ({n_changed} raw diff lines, expected -- reformatting only)")
    assert same, f"{fname}: extracted data differs -- investigate before promoting!"


IPscore_IT.html: data identical=True  (0 raw diff lines, expected -- reformatting only)


IPscore_IT_Demo.html: data identical=True  (0 raw diff lines, expected -- reformatting only)


IPscore_EN_Demo.html: data identical=True  (0 raw diff lines, expected -- reformatting only)


NPV_Target_Planner_EN.html: data identical=True  (0 raw diff lines, expected -- reformatting only)


NPV_Target_Planner_IT.html: data identical=True  (0 raw diff lines, expected -- reformatting only)


## 6. Verify — functional smoke test

Runs `buildPages()` + `fillDemo()` (or a deterministic score selection for the blank form)
in a sandboxed Node VM for both the **original** and **generated** file, and compares the
computed IPscore/NPV/verdict. This is the real correctness check — a text diff can't tell
you whether the *engine* still behaves the same; running it can.


In [7]:
def run_smoke_test(script_name, path):
    r = subprocess.run(["node", script_name, str(path)], cwd=BUILD, capture_output=True, text=True)
    return r.stdout.strip(), r.stderr.strip()

print("--- IPscore family ---")
for fname in ["IPscore_IT.html", "IPscore_IT_Demo.html", "IPscore_EN_Demo.html"]:
    live_out, _ = run_smoke_test("smoke_test.js", Path("..") / fname)
    gen_out, _ = run_smoke_test("smoke_test.js", DIST / fname)
    match = "MATCH" if live_out == gen_out else "MISMATCH"
    print(f"{fname}: {match}  live={live_out}  generated={gen_out}")


--- IPscore family ---


IPscore_IT.html: MATCH  live={"total":"0/200","pct":"0%","vanTotal":"0 €"}  generated={"total":"0/200","pct":"0%","vanTotal":"0 €"}


IPscore_IT_Demo.html: MATCH  live={"total":"0/200","pct":"0%","vanTotal":"0 €"}  generated={"total":"0/200","pct":"0%","vanTotal":"0 €"}


IPscore_EN_Demo.html: MATCH  live={"total":"0/200","pct":"0%","vanTotal":"€0"}  generated={"total":"0/200","pct":"0%","vanTotal":"€0"}


In [8]:
# NPV planner smoke test is inline (fillDemo + updateResults), since it needs an extra
# alert() stub not used by the IPscore engine.
npv_smoke_js = r'''
const fs=require("fs"), vm=require("vm");
function test(path){
  const html=fs.readFileSync(path,"utf8");
  const script=html.match(/<script>([\s\S]*?)<\/script>/)[1];
  function noop(){return undefined;}
  const elementsById=new Map();
  function makeEl(id){
    const el={className:"",style:{},innerHTML:"",checked:false,textContent:"",
      classList:{add:noop,remove:noop,toggle:noop,contains:()=>false},
      addEventListener:noop,appendChild:noop,querySelectorAll:()=>[],querySelector:()=>null};
    let v=""; Object.defineProperty(el,"value",{get(){return v;},set(x){v=x;}});
    return el;
  }
  function getElementById(id){ if(!elementsById.has(id)) elementsById.set(id, makeEl(id)); return elementsById.get(id); }
  const pagesDiv={appendChild:noop,innerHTML:""};
  const sandbox={
    document:{addEventListener:noop, getElementById:(id)=> id==="pages"?pagesDiv: id==="step-bar"?makeEl("step-bar"):getElementById(id),
      querySelector:()=>({checked:false,classList:{add:noop,remove:noop,toggle:noop}}), querySelectorAll:()=>[],
      createElement:()=>makeEl(Symbol("a"))},
    console, setTimeout:(fn)=>fn(), alert:()=>{}, Math,Object,Array,JSON,String,Number,Boolean,Date
  };
  sandbox.window=sandbox;
  vm.createContext(sandbox);
  vm.runInContext(script, sandbox, {filename:path, timeout:5000});
  sandbox.buildPages();
  sandbox.fillDemo();
  sandbox.updateResults();
  return JSON.stringify({npv:getElementById("res-npv-val").textContent,
    verdict:getElementById("res-verdict").innerHTML.slice(0,50)});
}
console.log(test(process.argv[2]));
'''
(BUILD / "_npv_smoke.js").write_text(npv_smoke_js, encoding="utf-8")

print("--- NPV Planner family ---")
for fname in ["NPV_Target_Planner_EN.html", "NPV_Target_Planner_IT.html"]:
    live_out, _ = run_smoke_test("_npv_smoke.js", Path("..") / fname)
    gen_out, _ = run_smoke_test("_npv_smoke.js", DIST / fname)
    match = "MATCH" if live_out == gen_out else "MISMATCH"
    print(f"{fname}: {match}  live={live_out}  generated={gen_out}")


--- NPV Planner family ---


NPV_Target_Planner_EN.html: MATCH  live={"npv":"€1,225,802","verdict":"<strong>&#10003; Target reached.</strong> Your cur"}  generated={"npv":"€1,225,802","verdict":"<strong>&#10003; Target reached.</strong> Your cur"}


NPV_Target_Planner_IT.html: MATCH  live={"npv":"1.225.802 €","verdict":"<strong>&#10003; Obiettivo raggiunto.</strong> Il "}  generated={"npv":"1.225.802 €","verdict":"<strong>&#10003; Obiettivo raggiunto.</strong> Il "}


## 6b. Verify — cross-check the NPV engine against the source Excel

The checks above only confirm the regenerated HTML behaves identically to the
*previous* HTML — they can't catch a bug that was already present in both. This step
is the actual algorithm-derivation check: it reads `IPscore_3.01 WORKHORSE.xlsx`
directly (the `Financial results` / `Financial calculations` sheets) and runs the
real `calcVAN()` / `calcNPV()` out of the rendered HTML against Excel's own 3
built-in test patents and their Excel-computed NPVs.

**History**: this check caught a real bug on 2026-07-16 — the `avgRev` window in
both engines summed exactly `depPeriod` years starting at `yFirst`, one year short of
Excel's actual `(T, T+1+depPeriod]` window (Excel always spans `depPeriod+1`
calendar-year slots). Invisible on the original demo patents (their commercial-lifetime
score was short enough that the extra year was always zero revenue), it silently
mis-stated NPV whenever lifetime is long relative to the depreciation period. Fixed in
both `IPscore_IT.html` and `NPV_Target_Planner_EN.html` (the two source-of-truth files
`make_*_template.py` transplant engine JS from), then propagated everywhere by
re-running the template generators + this render/promote pipeline.


In [9]:
r = subprocess.run([sys.executable, "verify_against_excel.py"], cwd=BUILD,
                   capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
assert r.returncode == 0, "Engine no longer matches the Excel source — do not promote"


=== Check 1: IPscore calcVAN() vs. Excel's 3 built-in test patents ===
  Patent 1: computed=329059.4284 expected=329059.4284 diff=0.000000 [PASS]
  Patent 2: computed=4361.2849 expected=4361.2849 diff=0.000000 [PASS]
  Patent 3: computed=-4686.3598 expected=-4686.3598 diff=0.000000 [PASS]

=== Check 2: NPV Planner calcNPV() vs. independent Python reference ===
  NPV Planner demo (NovaMed): computed=1225801.6019 expected=1225801.6019 diff=0.000000 [PASS]

ALL CHECKS PASSED



## 7. Promote — copy `build/dist/*.html` over the live files

**Manual, explicit step.** Only run this after the checks above are all green *and* you've
opened at least one generated file in an actual browser to confirm it looks/behaves right
(the Node smoke tests exercise the JS engine, not rendering/CSS/click-through UX).


In [10]:
PROMOTE = False  # flip to True and re-run this cell when you're ready

if PROMOTE:
    for fname in outputs:
        target = Path.cwd() / fname
        target.write_text((DIST / fname).read_text(encoding="utf-8"), encoding="utf-8")
        print(f"promoted build/dist/{fname} -> {fname}")
else:
    print("PROMOTE is False — nothing copied. Flip to True once you've reviewed build/dist/.")


PROMOTE is False — nothing copied. Flip to True once you've reviewed build/dist/.


## Extending this

- **New demo scenario** (e.g. a third fake patent case): copy one of the
  `build/data/ipscore_demo_*.json` / `npv_demo_*.json` files, edit the values, add a new
  `render_ipscore("it", demo_flag=True)`-style call with a new data pack name.
- **New language**: copy `ipscore_ui_strings_en.json` (and `ipscore_questions_en.json`,
  `npv_ui_strings_en.json`, `npv_insights_en.json`) to a new `_xx.json` suffix, translate
  every value, and render with that language code — no template changes needed. As a bonus,
  this pipeline can already produce a blank `IPscore_EN.html` for free
  (`render_ipscore("en", demo_flag=False)`), which doesn't exist as a live file today.
- **Fixing a bug that turns out to be structural** (not just content): edit the relevant
  `make_*_template.py` generator, re-run it, re-render, re-verify — never hand-edit the
  `.j2` file directly, or the next regeneration from the generator script will silently
  overwrite your fix.
